## Daily property health snapshot
Writes one row per property per `snapshot_date` to `<catalog>.<schema>.property_health_daily`, from `samples.wanderbricks.reviews`.

* Deleted reviews (`is_deleted = true`) are excluded from rating metrics.
* **is_declining**: ≥ 4 months with reviews and the monthly average rating fell in each of the last 3 month-over-month comparisons (months with no reviews are skipped). Same definition as `baseline_property_health`.
* Idempotent: reruns for the same `snapshot_date` replace that day's rows (`INSERT ... REPLACE WHERE`).
* Parameters: `catalog`, `schema`, `snapshot_date` (blank = today).

In [0]:
import datetime

dbutils.widgets.text("catalog", "genie_code_workshop")
dbutils.widgets.text("schema", "asml_dry_run")
dbutils.widgets.text("snapshot_date", "")  # blank = today (UTC)

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
snap_raw = dbutils.widgets.get("snapshot_date").strip()
snapshot_date = datetime.date.fromisoformat(snap_raw) if snap_raw else datetime.datetime.now(datetime.timezone.utc).date()

target = f"{catalog}.{schema}.property_health_daily"
source = "samples.wanderbricks.reviews"
print(f"target={target}  snapshot_date={snapshot_date}")

spark.sql("""
CREATE TABLE IF NOT EXISTS IDENTIFIER(:target) (
  snapshot_date          DATE      COMMENT 'Date this snapshot was computed for',
  property_id            BIGINT    COMMENT 'Property being reviewed',
  total_reviews          BIGINT    COMMENT 'Non-deleted reviews to date',
  deleted_reviews        BIGINT    COMMENT 'Deleted reviews to date',
  avg_rating             DOUBLE    COMMENT 'Average rating over all non-deleted reviews (1.0-5.0)',
  pct_low_ratings        DOUBLE    COMMENT 'Percent of non-deleted reviews rated below 2.0',
  first_review_at        TIMESTAMP COMMENT 'Earliest review',
  last_review_at         TIMESTAMP COMMENT 'Latest review',
  active_months          BIGINT    COMMENT 'Months with at least one non-deleted review',
  latest_month           DATE      COMMENT 'Most recent month with reviews',
  latest_month_avg       DOUBLE    COMMENT 'Average rating in latest_month',
  prev_month_avg         DOUBLE    COMMENT 'Average rating in the prior month with reviews',
  mom_change             DOUBLE    COMMENT 'latest_month_avg - prev_month_avg',
  declines_in_last_3     INT       COMMENT 'How many of the last 3 month-over-month changes were declines',
  decline_drop           DOUBLE    COMMENT 'For declining properties: latest_month_avg minus the average 3 active months earlier',
  is_declining           BOOLEAN   COMMENT '>= 4 active months and last 3 month-over-month changes all declines',
  computed_at            TIMESTAMP COMMENT 'When this row was written'
)
COMMENT 'Daily per-property review health snapshot built from samples.wanderbricks.reviews by the property_health_daily notebook'
""", args={"target": target})

In [0]:
# Reviews created on or before snapshot_date only, so backfills are reproducible.
# REPLACE WHERE overwrites just this snapshot_date, so reruns are idempotent.
spark.sql("""
INSERT INTO IDENTIFIER(:target) REPLACE WHERE snapshot_date = :snap
WITH src AS (
  SELECT * FROM IDENTIFIER(:source)
  WHERE created_at < date_add(:snap, 1)
),
prop AS (
  SELECT property_id,
         count_if(NOT is_deleted)                                   AS total_reviews,
         count_if(is_deleted)                                       AS deleted_reviews,
         avg(CASE WHEN NOT is_deleted THEN rating END)              AS avg_rating,
         100 * count_if(NOT is_deleted AND rating < 2) / nullif(count_if(NOT is_deleted), 0) AS pct_low_ratings,
         min(created_at)                                            AS first_review_at,
         max(created_at)                                            AS last_review_at
  FROM src GROUP BY property_id
),
monthly AS (
  SELECT property_id, CAST(date_trunc('month', created_at) AS DATE) AS month, avg(rating) AS avg_rating
  FROM src WHERE NOT is_deleted GROUP BY ALL
),
lagged AS (
  SELECT *,
         lag(avg_rating) OVER (PARTITION BY property_id ORDER BY month)      AS prev_avg,
         row_number()    OVER (PARTITION BY property_id ORDER BY month DESC) AS rn_desc,
         count(*)        OVER (PARTITION BY property_id)                     AS active_months
  FROM monthly
),
trend AS (
  SELECT property_id,
         max(active_months)                                          AS active_months,
         max(month) FILTER (WHERE rn_desc = 1)                       AS latest_month,
         max(avg_rating) FILTER (WHERE rn_desc = 1)                  AS latest_month_avg,
         max(prev_avg)   FILTER (WHERE rn_desc = 1)                  AS prev_month_avg,
         CAST(count_if(rn_desc <= 3 AND avg_rating < prev_avg) AS INT) AS declines_in_last_3,
         max(avg_rating) FILTER (WHERE rn_desc = 4)                  AS avg_4_back
  FROM lagged GROUP BY property_id
)
SELECT
  CAST(:snap AS DATE)                                   AS snapshot_date,
  p.property_id,
  p.total_reviews,
  p.deleted_reviews,
  round(p.avg_rating, 3)                                AS avg_rating,
  round(p.pct_low_ratings, 2)                           AS pct_low_ratings,
  p.first_review_at,
  p.last_review_at,
  coalesce(t.active_months, 0)                          AS active_months,
  t.latest_month,
  round(t.latest_month_avg, 3)                          AS latest_month_avg,
  round(t.prev_month_avg, 3)                            AS prev_month_avg,
  round(t.latest_month_avg - t.prev_month_avg, 3)       AS mom_change,
  coalesce(t.declines_in_last_3, 0)                     AS declines_in_last_3,
  CASE WHEN t.active_months >= 4 AND t.declines_in_last_3 = 3
       THEN round(t.latest_month_avg - t.avg_4_back, 3) END AS decline_drop,
  coalesce(t.active_months >= 4 AND t.declines_in_last_3 = 3, false) AS is_declining,
  current_timestamp()                                   AS computed_at
FROM prop p
LEFT JOIN trend t USING (property_id)
""", args={"target": target, "source": source, "snap": snapshot_date})

In [0]:
check = spark.sql("""
SELECT count(*)                    AS rows_written,
       count(DISTINCT property_id) AS distinct_properties,
       count_if(is_declining)      AS declining_properties,
       round(avg(avg_rating), 3)   AS mean_property_rating
FROM IDENTIFIER(:target) WHERE snapshot_date = :snap
""", args={"target": target, "snap": snapshot_date}).first()

expected = spark.sql(
    "SELECT count(DISTINCT property_id) AS n FROM IDENTIFIER(:source) WHERE created_at < date_add(:snap, 1)",
    args={"source": source, "snap": snapshot_date},
).first()["n"]

print(check.asDict(), "expected_properties=", expected)
assert check.rows_written > 0, "No rows written"
assert check.rows_written == check.distinct_properties, "Duplicate property rows for this snapshot_date"
assert check.rows_written == expected, f"Expected {expected} properties, wrote {check.rows_written}"